In [ ]:
import os
import json
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import rsa, padding as asym_padding
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives import padding as sym_padding


ecg_data = [0.5, 0.52, 0.48, 0.9, 0.1, -0.2, 0.5, 0.51, 0.49, 0.5]
# Convert data to bytes
data_bytes = json.dumps(ecg_data).encode()

private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
public_key = private_key.public_key()

# Layer 1: Symmetric layer
session_key = os.urandom(32)
iv = os.urandom(16)

padder = sym_padding.PKCS7(128).padder()
padded_data = padder.update(data_bytes) + padder.finalize()

cipher = Cipher(algorithms.AES(session_key), modes.CBC(iv))
encryptor = cipher.encryptor()
ciphertext = encryptor.update(padded_data) + encryptor.finalize()

# Layer 2: Asymmetric layer
wrapped_key = public_key.encrypt(
    session_key,
    asym_padding.OAEP(
        mgf=asym_padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

# Layer 3: Integrity Layer
package_to_sign = wrapped_key + iv + ciphertext
signature = private_key.sign(
    package_to_sign,
    asym_padding.PSS(
        mgf=asym_padding.MGF1(hashes.SHA256()),
        salt_length=asym_padding.PSS.MAX_LENGTH
    ),
    hashes.SHA256()
)

vault = {
    "wrapped_key": wrapped_key,
    "iv": iv,
    "ciphertext": ciphertext,
    "signature": signature
}

In [ ]:
try:
    # Verify Integrity
    public_key.verify(
        vault["signature"],
        vault["wrapped_key"] + vault["iv"] + vault["ciphertext"],
        asym_padding.PSS(
            mgf=asym_padding.MGF1(hashes.SHA256()),
            salt_length=asym_padding.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )
    print("Integrity Verified")

    # Unwrap AES Key
    decrypted_session_key = private_key.decrypt(
        vault["wrapped_key"],
        asym_padding.OAEP(
            mgf=asym_padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )

    # Decrypt the Data
    decryptor_cipher = Cipher(algorithms.AES(decrypted_session_key), modes.CBC(vault["iv"]))
    decryptor = decryptor_cipher.decryptor()
    decrypted_padded_data = decryptor.update(vault["ciphertext"]) + decryptor.finalize()

    # Remove padding and restoring original format
    unpadder = sym_padding.PKCS7(128).unpadder()
    final_data_bytes = unpadder.update(decrypted_padded_data) + unpadder.finalize()
    final_ecg_data = json.loads(final_data_bytes.decode())

    print(f"Decrypted Data: {final_ecg_data}")

except Exeception as e:
    print(f"Integrity not Verified. Either vault or key is wrong. {e}")
    

Integrity Verified
Decrypted Data: [0.5, 0.52, 0.48, 0.9, 0.1, -0.2, 0.5, 0.51, 0.49, 0.5]


In [8]:
import copy
from cryptography.exceptions import InvalidSignature

# Create a 'tampered' version of your vault
tampered_vault = copy.deepcopy(vault)

# --- SCENARIO A: Change one byte of the Encrypted Data ---
# We will change the very first byte of the ciphertext
original_byte = tampered_vault["ciphertext"][0]
# If the byte was 0, make it 1. Otherwise, make it 0.
tampered_byte = bytes([1]) if original_byte == 0 else bytes([0])
tampered_vault["ciphertext"] = tampered_byte + tampered_vault["ciphertext"][1:]

print("--- Testing Tampered Data ---")

try:
    # Verify Integrity
    public_key.verify(
        tampered_vault["signature"],
        tampered_vault["wrapped_key"] + tampered_vault["iv"] + tampered_vault["ciphertext"],
        asym_padding.PSS(
            mgf=asym_padding.MGF1(hashes.SHA256()),
            salt_length=asym_padding.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )
    print("Integrity Verified")

    # Unwrap AES Key
    decrypted_session_key = private_key.decrypt(
        tampered_vault["wrapped_key"],
        asym_padding.OAEP(
            mgf=asym_padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )

    # Decrypt the Data
    decryptor_cipher = Cipher(algorithms.AES(decrypted_session_key), modes.CBC(tampered_vault["iv"]))
    decryptor = decryptor_cipher.decryptor()
    decrypted_padded_data = decryptor.update(tampered_vault["ciphertext"]) + decryptor.finalize()

    # Remove padding and restoring original format
    unpadder = sym_padding.PKCS7(128).unpadder()
    final_data_bytes = unpadder.update(decrypted_padded_data) + unpadder.finalize()
    final_ecg_data = json.loads(final_data_bytes.decode())

    print(f"Decrypted Data: {final_ecg_data}")

except InvalidSignature:
    # This is a specific security error
    print("Integrity not Verified. Either vault or key is wrong.")

except Exception as e:
    print(f"An error occurred: {e}")
    

--- Testing Tampered Data ---
Integrity not Verified. Either vault or key is wrong.
